In [2]:
import abc
import collections
import enum
import math
import pathlib
import typing
import warnings

import numpy as np
import torch
import torch.optim
import torch.utils.data
import tqdm
from matplotlib import pyplot as plt

from util import draw_reliability_diagram, cost_function, setup_seeds, calc_calibration_curve

data_dir = pathlib.Path.cwd()
model_dir = pathlib.Path.cwd()
output_dir = pathlib.Path.cwd()

# Load training data
train_xs = torch.from_numpy(np.load(data_dir / "train_xs.npz")["train_xs"]) 
raw_train_meta = np.load(data_dir / "train_ys.npz")
train_ys = torch.from_numpy(raw_train_meta["train_ys"])
train_is_snow = torch.from_numpy(raw_train_meta["train_is_snow"])
train_is_cloud = torch.from_numpy(raw_train_meta["train_is_cloud"])
# 1800 samples (exact one of six), but includes info about snow or cloud
# Used for training with SWAG
dataset_train = torch.utils.data.TensorDataset(train_xs, train_is_snow, train_is_cloud, train_ys)

# Load validation data
val_xs = torch.from_numpy(np.load(data_dir / "val_xs.npz")["val_xs"])
raw_val_meta = np.load(data_dir / "val_ys.npz")
val_ys = torch.from_numpy(raw_val_meta["val_ys"])
val_is_snow = torch.from_numpy(raw_val_meta["val_is_snow"])
val_is_cloud = torch.from_numpy(raw_val_meta["val_is_cloud"])
# 140 samples (20 for each class {0,...,5} + 20 ambiguous images {-1})
# For calibration, not for SWAG
dataset_val = torch.utils.data.TensorDataset(val_xs, val_is_snow, val_is_cloud, val_ys)

# Fix all randomness
setup_seeds()

# Build and run the actual solution
train_loader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=16,
    shuffle=True,
    num_workers=0,
)

In [18]:
val_ys

tensor([ 2,  1,  3, -1,  1,  0,  5,  4, -1, -1,  0,  2,  5,  2,  5,  1,  3,  3,
         0, -1,  4,  2, -1,  4,  3, -1,  0,  1,  5,  2, -1,  4,  0,  5,  1,  3,
         3,  2,  5,  1, -1,  4,  3,  0,  2,  4,  0,  3, -1,  5,  1, -1,  2,  4,
         1,  1,  5,  2,  3,  0, -1,  2, -1,  1,  3, -1,  1, -1,  2,  1,  0,  3,
         4,  1,  3,  5,  5,  3,  0,  5,  4,  1,  5,  2,  4,  3,  0,  4,  0, -1,
         4,  4,  2,  1,  5,  4,  3,  5, -1, -1,  4,  0, -1,  1,  0,  2, -1,  0,
         1,  3,  0,  5,  5,  4,  5,  0,  1,  4,  2,  3,  0,  2,  4, -1,  3,  2,
         5,  2,  1,  5,  3,  4,  2,  0,  3,  1,  2,  5,  4,  0])

In [13]:
0%2

0